# Phase 1 baseline — Kaggle CPU load test

Reproducible baseline for the VinFast Digital Showroom backend (Phase 1).

**Fixed protocol** (see `docs/KAGGLE_LOAD_TEST.md`):

- Kaggle Notebook, *Accelerator: None* (CPU only), *Internet: On* (to clone the repository).
- One uvicorn worker, same command as the Docker image, SQLite database seeded from `data/catalog/vehicles.json`.
- Warm-up 30 s (discarded) → **3 runs × 120 s**, **concurrency 20**, read-only endpoints.
- Raw JSON for every run, environment metadata and a median summary are written to `/kaggle/working/results/`.

After the run: download `phase1-baseline-results.zip` (Output panel), unzip into
`loadtest/results/phase1-baseline/` in the repository, commit, then tag `phase1-baseline-v1`.

In [ ]:
# ---- Configuration: change only REPO_URL / GIT_REF ----
REPO_URL = "https://github.com/<owner>/<public-repo>.git"
GIT_REF = "main"            # branch, tag or commit hash under test

LABEL = "baseline"
CONCURRENCY = 20
WARMUP_SECONDS = 30
RUN_SECONDS = 120
RUNS = 3
PORT = 8000

In [ ]:
import json, os, platform, secrets, shutil, subprocess, sys, threading, time, urllib.request
from datetime import datetime, timezone
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = WORK / "repo"
RESULTS = WORK / "results"
if REPO.exists():
    shutil.rmtree(REPO)
RESULTS.mkdir(parents=True, exist_ok=True)

subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", GIT_REF], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Testing commit", COMMIT)

In [ ]:
BACKEND = REPO / "backend"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt", "-c", "constraints.txt"],
    cwd=BACKEND, check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "psutil"], check=True)
import psutil

## Environment metadata

In [ ]:
def cpu_model() -> str:
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"

DB_PATH = WORK / "baseline.db"
if DB_PATH.exists():
    DB_PATH.unlink()

SERVER_CMD = [sys.executable, "-m", "uvicorn", "src.main:app", "--host", "127.0.0.1", "--port", str(PORT)]
SERVER_ENV = {
    **os.environ,
    "APP_ENV": "production",
    "SECRET_KEY": secrets.token_urlsafe(48),
    "CORS_ORIGINS": "http://localhost",
    "DATABASE_URL": f"sqlite+aiosqlite:///{DB_PATH}",
    "STATIC_DIR": str(REPO / "static"),
    "SEED_DATA_DIR": str(REPO / "data" / "catalog"),
    "LOG_LEVEL": "WARNING",
}

metadata = {
    "label": LABEL,
    "measured_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "repository": REPO_URL,
    "git_ref": GIT_REF,
    "commit": COMMIT,
    "platform": platform.platform(),
    "python": platform.python_version(),
    "cpu_model": cpu_model(),
    "logical_cpus": os.cpu_count(),
    "physical_cpus": psutil.cpu_count(logical=False),
    "ram_gb": round(psutil.virtual_memory().total / 1024**3, 2),
    "kaggle_kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "not-kaggle"),
    "database": "sqlite (aiosqlite), file on local disk",
    "dataset": "data/catalog/vehicles.json + built-in locations dataset",
    "server_command": " ".join(SERVER_CMD[1:]),
    "workers": 1,
    "concurrency": CONCURRENCY,
    "warmup_seconds": WARMUP_SECONDS,
    "run_seconds": RUN_SECONDS,
    "runs": RUNS,
}
metadata

## Start the backend

In [ ]:
server_log = open(RESULTS / "server.log", "w")
server = subprocess.Popen(SERVER_CMD, cwd=BACKEND, env=SERVER_ENV, stdout=server_log, stderr=subprocess.STDOUT)
BASE_URL = f"http://127.0.0.1:{PORT}"

for _ in range(120):
    try:
        with urllib.request.urlopen(f"{BASE_URL}/health", timeout=2) as response:
            health = json.load(response)
            if health.get("database") == "connected":
                break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError("backend did not become healthy; see results/server.log")

def count(path):
    with urllib.request.urlopen(f"{BASE_URL}{path}", timeout=10) as response:
        body = json.load(response)
    return body["total"] if isinstance(body, dict) else len(body)

metadata["dataset_rows"] = {
    "cars": count("/api/v1/cars"),
    "motorbikes": count("/api/v1/motorbikes"),
    "locations": count("/api/v1/locations"),
}
(RESULTS / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(health, metadata["dataset_rows"])

## Warm-up, then three measured runs

In [ ]:
def run_load(name: str, seconds: int) -> dict:
    output = RESULTS / f"{name}.json"
    process = psutil.Process(server.pid)
    process.cpu_percent(None)
    samples, stop = [], threading.Event()

    def sample():
        while not stop.wait(1.0):
            samples.append((process.cpu_percent(None), process.memory_info().rss / 1024**2))

    sampler = threading.Thread(target=sample, daemon=True)
    sampler.start()
    completed = subprocess.run(
        [sys.executable, "scripts/load_test.py", "--base-url", BASE_URL,
         "--duration", str(seconds), "--concurrency", str(CONCURRENCY), "--output", str(output)],
        cwd=BACKEND, capture_output=True, text=True,
    )
    stop.set()
    sampler.join()
    if not output.exists():
        raise RuntimeError(completed.stderr or completed.stdout)

    result = json.loads(output.read_text(encoding="utf-8"))
    if samples:
        cpu = [value for value, _ in samples]
        result["server_resources"] = {
            "cpu_percent_mean": round(sum(cpu) / len(cpu), 1),
            "cpu_percent_max": round(max(cpu), 1),
            "rss_mb_max": round(max(rss for _, rss in samples), 1),
            "samples": len(samples),
        }
    result["commit"] = COMMIT
    output.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    print(f"{name}: {result['requests_per_second']} rps, p95 {result['latency_ms']['p95']} ms, "
          f"errors {result['error_rate_percent']}%")
    return result

run_load(f"{LABEL}-warmup", WARMUP_SECONDS)
for index in range(1, RUNS + 1):
    run_load(f"{LABEL}-run-{index}", RUN_SECONDS)

## Summary

In [ ]:
run_files = [str(RESULTS / f"{LABEL}-run-{index}.json") for index in range(1, RUNS + 1)]
summary = subprocess.run(
    [sys.executable, "scripts/summarize_load_tests.py", *run_files,
     "--label", LABEL.capitalize(), "--metadata", str(RESULTS / "metadata.json"),
     "--output", str(RESULTS / f"{LABEL}-summary")],
    cwd=BACKEND, capture_output=True, text=True, check=True,
)
from IPython.display import Markdown, display
display(Markdown(summary.stdout))

In [ ]:
server.terminate()
server.wait(timeout=30)
server_log.close()
archive = shutil.make_archive(str(WORK / "phase1-baseline-results"), "zip", RESULTS)
print("Download:", archive)
sorted(path.name for path in RESULTS.iterdir())